In [0]:
CREATE WIDGET TEXT catalog_name DEFAULT "mk31_dev";

In [0]:
%sql
USE CATALOG :catalog_name;
USE SCHEMA gold;

In [0]:
CREATE OR REPLACE VIEW gold.vw_magnitude_segmentation AS
SELECT
    CASE
        WHEN mag < 2.0 THEN '1. Micro (< 2.0)'
        WHEN mag >= 2.0 AND mag < 4.0 THEN '2. Minor (2.0 - 3.9)'
        WHEN mag >= 4.0 AND mag < 6.0 THEN '3. Moderate (4.0 - 5.9)'
        WHEN mag >= 6.0 AND mag < 8.0 THEN '4. Strong (6.0 - 7.9)'
        WHEN mag >= 8.0 THEN '5. Major (8.0+)'
        ELSE 'Unknown'
    END AS severity_bucket,
    COUNT(*) AS earthquake_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM silver.earthquake_data_final
WHERE mag IS NOT NULL
GROUP BY severity_bucket
ORDER BY severity_bucket;


In [0]:
CREATE OR REPLACE VIEW gold.vw_earthquake_map AS
SELECT
    id,
    place,
    latitude,
    longitude,
    depth,
    mag,
    time,
    CASE 
        WHEN tsunami = 1.0 THEN 'Tsunami' 
        ELSE 'No Tsunami' 
    END AS tsunami_label,
    tsunami
FROM silver.earthquake_data_final
WHERE latitude IS NOT NULL 
  AND longitude IS NOT NULL;

In [0]:
CREATE OR REPLACE VIEW gold.vw_frequent_areas AS
SELECT
    TRIM(SPLIT(place, ',')[SIZE(SPLIT(place, ',')) - 1]) AS region,
    COUNT(*) AS total_count,
    COUNT(CASE WHEN time >= CURRENT_TIMESTAMP - INTERVAL 7 DAYS 
               THEN 1 END)                               AS last_7_days_count,
    ROUND(AVG(mag), 2)                                   AS avg_magnitude
FROM silver.earthquake_data_final
GROUP BY region
ORDER BY last_7_days_count DESC
LIMIT 20;